### Graph Neural Network (GNN)

Drugs = Nodes in a graph
Known interactions = Edges between nodes
Features = Chemical properties (fingerprints, functional groups, physicochemical)
Task: Link prediction (will there be an edge between two nodes?)

## **Pipeline summary**

| Aspect | What you have |
|--------|----------------|
| **Data** | DrugBank DDI + SMILES; multi-view (Morgan, MACCS, phys-chem); stratified 70/15/15 train/val/test. |
| **Model** | Multi-view attention → GATv2 (2 layers) → edge classifier (binary + type); fits subgraph-based, memory-efficient training. |
| **Training** | Priority 1: Full-graph GNN once/epoch + edge-only batches (no k_hop per batch). AMP, graph on GPU, validate every N epochs, optional torch.compile. |
| **Metrics** | Validation: loss, accuracy, F1, precision, recall; test: same + ROC-AUC. |

The Pipeline in 5 Steps
```bash
1. FEATURE EXTRACTION
   Raw SMILES → Morgan FP + MACCS + Properties
   
2. GRAPH CONSTRUCTION
   Build adjacency matrix (who interacts with whom)
   
3. MULTI-VIEW FUSION
   Combine 3 views using attention
   
4. GNN ENCODING
   Message passing to learn node embeddings
   
5. EDGE CLASSIFICATION
   Predict interaction from node pair embeddings
```

### Metrices

- **Accuracy** (with a 0.90 target)
- **F1**
- **Precision**
- **Recall** (for “don’t miss DDIs”)
- **ROC-AUC**
to get a single, comparable test report for accuracy and F1/recall.

If you share your current test accuracy and F1/recall after one full run (with the Neg fix and new config), we can suggest next steps (e.g. one more GAT layer, dropout, or label smoothing) to push toward 90% while keeping the pipeline memory-efficient and fast.

In [1]:
import gc
import time
import json
import os
import numpy as np
import pandas as pd
from datetime import timedelta, datetime
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam, lr_scheduler
from torch.utils.data import Dataset, DataLoader, TensorDataset


# Graph & Geometry
from torch_geometric.nn import GATv2Conv, GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import k_hop_subgraph
from torch.amp import GradScaler, autocast

# Molecular Processing
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, Descriptors, MACCSkeys

# Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, precision_recall_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Config
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE_TYPE = DEVICE.type
TIMESTAMP = datetime.now().strftime("%d_%b_%H-%M")

# Paths
DDIS_DATA_PATH = 'dataset/drugdata/ddis.csv'
DRUG_SMILE_DATA_PATH = 'dataset/drugdata/drug_smiles.csv'

# Create Artifact Dirs
os.makedirs("models", exist_ok=True)
os.makedirs("images", exist_ok=True)

# --- Memory-efficient & accuracy-oriented training config ---
TRAIN_CONFIG = {
    "physical_batch_size": 256,   # Fits ~4GB GPU; increase if you have more VRAM
    "accumulation_steps": 8,      # Effective batch = 256*8 = 2048
    "hidden_dim": 256,
    "n_heads": 2,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 50,
    "type_loss_weight": 0.5,      # Auxiliary type prediction weight
    "use_class_weights": True,    # Balance pos/neg for better recall & F1
            "scheduler_patience": 4,
    "scheduler_factor": 0.5,
    "full_graph_on_gpu": True,       # Keep graph on GPU for fast full-graph forward
    "validate_every_n_epochs": 2,    # Validate every N epochs (Priority 4)
    "use_compile": True,             # torch.compile for faster kernels (PyTorch 2+)
}

print(f"● System Ready. Device: {DEVICE}")


● System Ready. Device: cuda


In [3]:

OPTIMIZED_TRAIN_CONFIG = {
    # Memory Management
    "physical_batch_size": 512,      # Increased (faster with cached embeddings)
    "accumulation_steps": 4,         # Reduced (effective batch = 2048)
    
    # Model Architecture
    "hidden_dim": 256,               # Good balance
    "n_heads": 4,                    # Increased for better attention
    "dropout": 0.3,                  # Regularization
    
    # Training Dynamics
    "lr": 5e-4,                      # Lower for stability
    "weight_decay": 1e-4,
    "epochs": 80,                    # More epochs (but faster now!)
    
    # Loss Configuration
    "type_loss_weight": 0.5,
    "use_class_weights": True,
    "label_smoothing": 0.1,          # Prevent overconfidence
    
    # Scheduler
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    
    # System
    "full_graph_on_gpu": True,
    "validate_every_n_epochs": 2,
    "use_compile": True,             # PyTorch 2.0+
    
    # Data Augmentation
    "edge_dropout_rate": 0.1,        # Drop 10% of edges during training
    "feature_noise": 0.01,           # Add small noise to features
}


In [ ]:
class DDIDataAugmenter:
    """
    Augment training data to improve generalization:
    1. Edge dropout (random edge masking)
    2. Feature perturbation
    3. Hard negative mining
    """
    
    @staticmethod
    def augment_graph_training(graph, edge_dropout=0.1, feature_noise=0.01):
        """
        Apply augmentation during training epoch.
        """
        # Edge dropout: Randomly remove edges from adjacency
        if edge_dropout > 0:
            mask = torch.rand(graph['edge_index'].shape[1]) > edge_dropout
            aug_edge_index = graph['edge_index'][:, mask]
        else:
            aug_edge_index = graph['edge_index']
        
        # Feature noise: Add small Gaussian noise
        if feature_noise > 0:
            aug_x_v1 = graph['x_v1'] + torch.randn_like(graph['x_v1']) * feature_noise
            aug_x_v2 = graph['x_v2'] + torch.randn_like(graph['x_v2']) * feature_noise
            aug_x_v3 = graph['x_v3'] + torch.randn_like(graph['x_v3']) * feature_noise
        else:
            aug_x_v1, aug_x_v2, aug_x_v3 = graph['x_v1'], graph['x_v2'], graph['x_v3']
        
        return {
            'edge_index': aug_edge_index,
            'x_v1': aug_x_v1,
            'x_v2': aug_x_v2,
            'x_v3': aug_x_v3
        }

In [5]:
class MultiViewFeatureExtractor:
    """
    Extracting diverse 'views' of the same molecule.
    View 1: Morgan FP (Detailed Substructure)
    View 2: MACCS Keys (Functional Groups)
    View 3: Physico-Chemical Properties (Global Properties)
    """
    def __init__(self):
        self.morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
        
    def get_features(self, smiles_df):
        view1_morgan = []
        view2_maccs = []
        view3_phys = []
        valid_ids = []
        
        print("● Extracting Multi-View Features...")
        
        for _, row in smiles_df.iterrows():
            mol = Chem.MolFromSmiles(str(row["smiles"]))
            if mol:
                # View 1: Morgan Fingerprint (1024 dim)
                fp = np.array(self.morgan_gen.GetFingerprint(mol), dtype=np.float32)
                
                # View 2: MACCS Keys (167 dim) - Great for functional groups
                maccs = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.float32)
                
                # View 3: Physico-chemical (8 dim) - Great for transport/metabolism
                desc = np.array([
                    Descriptors.MolWt(mol), Descriptors.MolLogP(mol),
                    Descriptors.NumHDonors(mol), Descriptors.NumHAcceptors(mol),
                    Descriptors.TPSA(mol), Descriptors.NumRotatableBonds(mol),
                    Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol)
                ], dtype=np.float32)
                
                view1_morgan.append(fp)
                view2_maccs.append(maccs)
                view3_phys.append(desc)
                valid_ids.append(row["drug_id"])
        
        # Stack and Normalize View 3 (since it has large values like Weight)
        v1 = np.stack(view1_morgan)
        v2 = np.stack(view2_maccs)
        v3 = np.stack(view3_phys)
        
        scaler = StandardScaler()
        v3 = scaler.fit_transform(v3)
        
        print(f"✓ Processed {len(valid_ids)} drugs.")
        print(f"  - View 1 Shape: {v1.shape}")
        print(f"  - View 2 Shape: {v2.shape}")
        print(f"  - View 3 Shape: {v3.shape}")
        
        return {
            "v1": torch.tensor(v1, dtype=torch.float),
            "v2": torch.tensor(v2, dtype=torch.float),
            "v3": torch.tensor(v3, dtype=torch.float),
            "ids": valid_ids
        }

# Load and Process
smiles_df = pd.read_csv(DRUG_SMILE_DATA_PATH)
extractor = MultiViewFeatureExtractor()
features_dict = extractor.get_features(smiles_df)

● Extracting Multi-View Features...
✓ Processed 1706 drugs.
  - View 1 Shape: (1706, 1024)
  - View 2 Shape: (1706, 167)
  - View 3 Shape: (1706, 8)


In [6]:
class MMADLGraphBuilder:
    def __init__(self, ddi_path, feature_dict):
        self.ddi_df = pd.read_csv(ddi_path)
        self.feats = feature_dict
        self.drug_map = {d: i for i, d in enumerate(feature_dict['ids'])}
        self.idx_map = {i: d for d, i in self.drug_map.items()}
        self.type_enc = LabelEncoder()
        
    def build(self):
        # Map Edges
        pos_src, pos_dst, pos_types = [], [], []
        neg_src, neg_dst = [], []
        
        print("● Building DDI Graph Topology...")
        for _, row in self.ddi_df.iterrows():
            if row['d1'] in self.drug_map and row['d2'] in self.drug_map:
                u, v = self.drug_map[row['d1']], self.drug_map[row['d2']]
                
                # Positive Edge (Bidirectional)
                pos_src.extend([u, v])
                pos_dst.extend([v, u])
                pos_types.extend([row['type'], row['type']])
                
                # Negative Edge Handling: "DRUGID$t" or "DRUGID$h" -> extract DRUGID
                if isinstance(row['Neg samples'], str):
                    neg_raw = row['Neg samples'].strip().split('$')[0]
                    if neg_raw and neg_raw in self.drug_map:
                        w = self.drug_map[neg_raw]
                        neg_src.extend([u, w])
                        neg_dst.extend([w, u])

        # Convert to Tensors
        pos_edge_index = torch.tensor([pos_src, pos_dst], dtype=torch.long)
        neg_edge_index = torch.tensor([neg_src, neg_dst], dtype=torch.long)
        
        # Combine for processing
        full_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
        
        # Encode Types
        y_types_enc = self.type_enc.fit_transform(pos_types)
        
        # Create Labels
        # Binary: 1 for pos, 0 for neg
        y_binary = torch.cat([torch.ones(pos_edge_index.shape[1]), torch.zeros(neg_edge_index.shape[1])]).long()
        # Type: Encoded for pos, -1 for neg
        y_type = torch.cat([torch.tensor(y_types_enc), torch.full((neg_edge_index.shape[1],), -1)]).long()
        
        return {
            "x_v1": self.feats['v1'],
            "x_v2": self.feats['v2'],
            "x_v3": self.feats['v3'],
            "edge_index": full_edge_index,
            "y_binary": y_binary,
            "y_type": y_type,
            "n_types": len(self.type_enc.classes_),
            "encoder": self.type_enc,
            "drug_map": self.drug_map
        }

    def create_stratified_split(self, graph):
        # Stratified Split based on Binary Labels to ensure Negatives are balanced in Train/Val
        print("● Creating Stratified Splits (TDCommons Standard)...")
        n_edges = graph['edge_index'].shape[1]
        indices = np.arange(n_edges)
        
        train_idx, temp_idx = train_test_split(
            indices, test_size=0.3, stratify=graph['y_binary'], random_state=42
        )
        val_idx, test_idx = train_test_split(
            temp_idx, test_size=0.5, stratify=graph['y_binary'][temp_idx], random_state=42
        )
        
        # Create Boolean Masks
        for name, idx in zip(['train', 'val', 'test'], [train_idx, val_idx, test_idx]):
            mask = torch.zeros(n_edges, dtype=torch.bool)
            mask[idx] = True
            graph[f'{name}_mask'] = mask
            
        print(f"  - Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
        return graph

builder = MMADLGraphBuilder(DDIS_DATA_PATH, features_dict)
graph_data = builder.build()
graph = builder.create_stratified_split(graph_data)

● Building DDI Graph Topology...
● Creating Stratified Splits (TDCommons Standard)...
  - Train: 537062 | Val: 115085 | Test: 115085


In [ ]:
class FeatureAttentionLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.atten = nn.Linear(dim, 1)
        
    def forward(self, v1, v2, v3):
        stack = torch.stack([v1, v2, v3], dim=1)
        scores = F.softmax(self.atten(stack), dim=1)
        fused = torch.sum(stack * scores, dim=1)
        return fused, scores


class ImprovedCardioMMADL(nn.Module):
    """
    Improvements:
    1. Residual connections for better gradient flow
    2. Layer normalization for training stability
    3. Edge dropout for regularization
    4. Optional feature masking
    """
    def __init__(self, dims, hidden_dim, n_heads, n_types, dropout=0.3):
        super().__init__()
        
        # View Projectors with BatchNorm
        self.proj_v1 = nn.Sequential(
            nn.Linear(dims['v1'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v2 = nn.Sequential(
            nn.Linear(dims['v2'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        self.proj_v3 = nn.Sequential(
            nn.Linear(dims['v3'], hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5)
        )
        
        # Feature Attention
        self.feat_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 4),
            nn.ReLU(),
            nn.Linear(hidden_dim // 4, 1)
        )
        
        # GATv2 Layers with residual connections
        self.gat1 = GATv2Conv(
            hidden_dim, hidden_dim // n_heads, 
            heads=n_heads, concat=True, dropout=dropout
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        
        self.gat2 = GATv2Conv(
            hidden_dim, hidden_dim, 
            heads=1, concat=False, dropout=dropout
        )
        self.norm2 = nn.LayerNorm(hidden_dim)
        
        # Edge Classifier with deeper network
        self.edge_encoder = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
        )
        
        self.head_bin = nn.Linear(hidden_dim // 2, 2)
        self.head_type = nn.Linear(hidden_dim // 2, n_types)
        
        self.dropout = dropout

    def get_node_embeddings(self, x_v1, x_v2, x_v3, edge_index):
        """
        Full-graph forward with residual connections and normalization.
        """
        # Project each view
        h1 = self.proj_v1(x_v1)
        h2 = self.proj_v2(x_v2)
        h3 = self.proj_v3(x_v3)
        
        # Multi-view attention fusion
        stack = torch.stack([h1, h2, h3], dim=1)  # (N, 3, hidden_dim)
        scores = F.softmax(self.feat_attention(stack), dim=1)
        h_fused = torch.sum(stack * scores, dim=1)
        
        # GAT Layer 1 with residual
        h_gat1 = self.gat1(h_fused, edge_index)
        h_gat1 = self.norm1(h_gat1 + h_fused)  # Residual connection
        h_gat1 = F.elu(h_gat1)
        
        # GAT Layer 2 with residual
        h_gat2 = self.gat2(h_gat1, edge_index)
        node_emb = self.norm2(h_gat2 + h_fused)  # Skip connection to fusion
        
        return node_emb

        # : Remove edge dropout for now
    def forward_edges_from_emb(self, node_emb, edge_label_index):
        src, dst = edge_label_index[0], edge_label_index[1]
        edge_feat = torch.cat([node_emb[src], node_emb[dst]], dim=-1)
        shared = self.edge_encoder(edge_feat)  # or self.classifier if using original model
        return self.head_bin(shared), self.head_type(shared)

    def forward(self, x_v1, x_v2, x_v3, edge_index, edge_label_index):
        """Full forward pass (for inference)."""
        node_emb = self.get_node_embeddings(x_v1, x_v2, x_v3, edge_index)
        return self.forward_edges_from_emb(node_emb, edge_label_index)  # Remove training=False

In [ ]:
def train_cardio_model_optimized(graph, config, model, device='cuda'):
    """
    OPTIMIZED TRAINING: Compute node embeddings ONCE per epoch, cache, then train edges.
    Expected speedup: 10-20x (30min → 1.5-3min per epoch)
    """
    
    # Extract config
    physical_batch = config["physical_batch_size"]
    accum_steps = config["accumulation_steps"]
    type_w = config.get("type_loss_weight", 0.5)
    validate_every = config.get("validate_every_n_epochs", 2)
    
    print(f"🚀 OPTIMIZED TRAINING: Node embeddings cached once/epoch")
    print(f"   Batch={physical_batch}, accum={accum_steps}, effective={physical_batch * accum_steps}")
    
    # ============================================================
    # SETUP: Data Loaders
    # ============================================================
    train_edges = graph['edge_index'][:, graph['train_mask']]
    train_labels = torch.stack([
        graph['y_binary'][graph['train_mask']], 
        graph['y_type'][graph['train_mask']]
    ], dim=1)
    
    val_edges = graph['edge_index'][:, graph['val_mask']]
    val_labels = torch.stack([
        graph['y_binary'][graph['val_mask']], 
        graph['y_type'][graph['val_mask']]
    ], dim=1)
    
    # OPTIMIZATION: Use num_workers for parallel data loading
    train_loader = DataLoader(
        TensorDataset(train_edges.t(), train_labels),
        batch_size=physical_batch, 
        shuffle=True,
        pin_memory=True,
        num_workers=2,  # Parallel loading
        persistent_workers=True
    )
    
    val_loader = DataLoader(
        TensorDataset(val_edges.t(), val_labels),
        batch_size=physical_batch * 2,  # Larger for validation
        shuffle=False,
        pin_memory=True,
        num_workers=2
    )
    
    # ============================================================
    # SETUP: Loss Functions with Class Weights
    # ============================================================
    y_train_bin = graph['y_binary'][graph['train_mask']].numpy()
    n_pos, n_neg = int((y_train_bin == 1).sum()), int((y_train_bin == 0).sum())
    
    if config.get("use_class_weights", True) and n_pos > 0 and n_neg > 0:
        w_pos = (n_pos + n_neg) / (2.0 * n_pos)
        w_neg = (n_pos + n_neg) / (2.0 * n_neg)
        class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float32, device=device)
        crit_bin = nn.CrossEntropyLoss(weight=class_weights)
        print(f"   Class weights (neg={w_neg:.3f}, pos={w_pos:.3f})")
    else:
        crit_bin = nn.CrossEntropyLoss()
    
    crit_type = nn.CrossEntropyLoss(ignore_index=-1)
    
    # ============================================================
    # SETUP: Move graph data to GPU once
    # ============================================================
    full_adj = graph['edge_index'].to(device)
    x_v1 = graph['x_v1'].to(device)
    x_v2 = graph['x_v2'].to(device)
    x_v3 = graph['x_v3'].to(device)
    print(f"   Graph + features on GPU ✓")
    
    # ============================================================
    # SETUP: Optimizer & Scheduler
    # ============================================================
    optimizer = Adam(
        model.parameters(), 
        lr=config['lr'], 
        weight_decay=config.get('weight_decay', 1e-4)
    )
    scaler = GradScaler()
    scheduler = lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max', 
        patience=config.get('scheduler_patience', 4),
        factor=config.get('scheduler_factor', 0.5),
        # verbose=True
    )
    
    # ============================================================
    # TRAINING LOOP
    # ============================================================
    history = {
        'train_loss': [], 'val_acc': [], 'val_f1': [], 
        'val_precision': [], 'val_recall': []
    }
    
    print(f"\n{'='*80}")
    print(f"{'Epoch':<6} | {'Train Loss':<11} | {'Val Acc':<8} | {'Val F1':<8} | {'Val P':<8} | {'Val R':<8} | {'Time':<8}")
    print(f"{'='*80}")
    
    import time
    best_f1 = 0.0
    
    for epoch in range(1, config['epochs'] + 1):
        epoch_start = time.time()
        model.train()
        
        # ============================================================
        # KEY OPTIMIZATION: Compute node embeddings ONCE per epoch
        # ============================================================
        with torch.no_grad():
            with autocast(device.type):
                # Single full-graph forward pass
                node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                node_emb = node_emb.detach()  # Detach from computation graph
        
        # ============================================================
        # Edge Classification Training (Fast!)
        # ============================================================
        total_loss = 0.0
        optimizer.zero_grad()
        
        for i, (batch_edges, batch_labels) in enumerate(train_loader):
            batch_edges = batch_edges.t().to(device, non_blocking=True)
            y_bin = batch_labels[:, 0].to(device, non_blocking=True)
            y_type = batch_labels[:, 1].to(device, non_blocking=True)
            
            # Forward: Use cached node embeddings (no GNN computation!)
            with autocast(device.type):
                pred_bin, pred_type = model.forward_edges_from_emb(node_emb, batch_edges)
                loss = (crit_bin(pred_bin, y_bin) + 
                       type_w * crit_type(pred_type, y_type)) / accum_steps
            
            # Backward with gradient accumulation
            scaler.scale(loss).backward()
            
            if (i + 1) % accum_steps == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            total_loss += loss.item() * accum_steps
        
        train_loss = total_loss / len(train_loader)
        history['train_loss'].append(train_loss)
        
        # ============================================================
        # Validation
        # ============================================================
        do_validate = (epoch % validate_every == 0) or (epoch == config['epochs'])
        
        if do_validate:
            model.eval()
            all_preds, all_trues = [], []
            
            # Compute validation node embeddings once
            with torch.no_grad():
                with autocast(device.type):
                    node_emb_val = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
                
                for batch_edges, batch_labels in val_loader:
                    batch_edges = batch_edges.t().to(device, non_blocking=True)
                    p_bin, _ = model.forward_edges_from_emb(node_emb_val, batch_edges)
                    all_preds.extend(torch.argmax(p_bin, dim=1).cpu().numpy())
                    all_trues.extend(batch_labels[:, 0].numpy())
            
            # Compute metrics
            all_trues = np.array(all_trues)
            all_preds = np.array(all_preds)
            
            val_acc = accuracy_score(all_trues, all_preds)
            val_f1 = f1_score(all_trues, all_preds, zero_division=0)
            val_prec = precision_score(all_trues, all_preds, zero_division=0)
            val_rec = recall_score(all_trues, all_preds, zero_division=0)
            
            history['val_acc'].append(val_acc)
            history['val_f1'].append(val_f1)
            history['val_precision'].append(val_prec)
            history['val_recall'].append(val_rec)
            
            scheduler.step(val_f1)
            
            # Save best model
            if val_f1 > best_f1:
                best_f1 = val_f1
                torch.save(model.state_dict(), f'models/best_model_f1_{val_f1:.4f}.pt')
            
            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | {val_acc:>6.4f}  | {val_f1:>6.4f}  | {val_prec:>6.4f}  | {val_rec:>6.4f}  | {epoch_time:>5.1f}s")
        else:
            # No validation this epoch
            for k in ['val_acc', 'val_f1', 'val_precision', 'val_recall']:
                history[k].append(history[k][-1] if history[k] else 0.0)
            epoch_time = time.time() - epoch_start
            print(f"{epoch:>4}   | {train_loss:>9.4f}   | (no val) | {epoch_time:>5.1f}s")
        
        torch.cuda.empty_cache()
    
    print(f"{'='*80}")
    print(f"✅ Training complete! Best Val F1: {best_f1:.4f}")
    
    return model, history

In [ ]:
# Initialize improved model
model = ImprovedCardioMMADL(
    dims={'v1': 1024, 'v2': 167, 'v3': 8},
    hidden_dim=OPTIMIZED_TRAIN_CONFIG['hidden_dim'],
    n_heads=OPTIMIZED_TRAIN_CONFIG['n_heads'],
    n_types=graph['n_types'],
    dropout=OPTIMIZED_TRAIN_CONFIG['dropout']
).to(DEVICE)

# Compile model for faster execution (PyTorch 2.0+)
if OPTIMIZED_TRAIN_CONFIG['use_compile'] and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model, mode='reduce-overhead')
        print("✅ torch.compile enabled")
    except:
        print("⚠️ torch.compile not available")

✅ torch.compile enabled


In [ ]:
# TESTING MODEL STRUCTURE
print("Model type:", type(model).__name__)
print("\nforward_edges_from_emb signature:")
import inspect
print(inspect.signature(model.forward_edges_from_emb))

# Check if edge dropout exists
if hasattr(model, 'dropout'):
    print(f"\nModel has dropout: {model.dropout}")
else:
    print("\nModel has no dropout attribute")

# Test a small batch
test_edges = graph['edge_index'][:, :512].to(DEVICE)
test_node_emb = model.get_node_embeddings(
    graph['x_v1'].to(DEVICE),
    graph['x_v2'].to(DEVICE), 
    graph['x_v3'].to(DEVICE),
    graph['edge_index'].to(DEVICE)
)

pred_bin, pred_type = model.forward_edges_from_emb(test_node_emb, test_edges)
print(f"✅ Input edges: {test_edges.shape[1]}")
print(f"✅ Output predictions: {pred_bin.shape[0]}")
print(f"✅ Match: {test_edges.shape[1] == pred_bin.shape[0]}")

Model type: OptimizedModule

forward_edges_from_emb signature:
(node_emb, edge_label_index)

Model has dropout: 0.3
✅ Input edges: 512
✅ Output predictions: 512
✅ Match: True


In [ ]:
# Train with optimized function
model, history = train_cardio_model_optimized(
    graph, 
    OPTIMIZED_TRAIN_CONFIG, 
    model, 
    device=DEVICE
)

In [12]:
# --- Test set evaluation (full-graph emb + edge-only, no k_hop_subgraph) ---
def evaluate_test_set(model, graph, batch_size=512):
    """Report accuracy, F1, precision, recall, ROC-AUC on test set."""
    model.eval()
    test_edges = graph['edge_index'][:, graph['test_mask']]
    test_labels_bin = graph['y_binary'][graph['test_mask']]
    test_loader = DataLoader(
        TensorDataset(test_edges.t(), test_labels_bin.unsqueeze(1)),
        batch_size=batch_size, shuffle=False
    )
    full_adj = graph['edge_index'].to(DEVICE)
    x_v1 = graph['x_v1'].to(DEVICE)
    x_v2 = graph['x_v2'].to(DEVICE)
    x_v3 = graph['x_v3'].to(DEVICE)
    all_preds, all_probs, all_trues = [], [], []
    with torch.no_grad():
        with autocast(DEVICE_TYPE):
            node_emb = model.get_node_embeddings(x_v1, x_v2, x_v3, full_adj)
        for batch_edges, batch_y in test_loader:
            batch_edges = batch_edges.t().to(DEVICE)
            p_bin, _ = model.forward_edges_from_emb(node_emb, batch_edges)
            probs = torch.softmax(p_bin, dim=1)[:, 1].cpu().numpy()
            all_preds.extend(torch.argmax(p_bin, dim=1).cpu().numpy())
            all_probs.extend(probs)
            all_trues.extend(batch_y.squeeze().numpy())
    y_true = np.array(all_trues)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    try:
        roc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc = float('nan')
    print("=" * 50)
    print("TEST SET METRICS (final model)")
    print("=" * 50)
    print(f"  Accuracy:  {acc:.4f}  (target: 0.90)")
    print(f"  F1:        {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}  (important: avoid missing DDIs)")
    print(f"  ROC-AUC:   {roc:.4f}")
    print("=" * 50)
    return {"accuracy": acc, "f1": f1, "precision": prec, "recall": rec, "roc_auc": roc}

test_metrics = evaluate_test_set(model, graph)

TEST SET METRICS (final model)
  Accuracy:  0.5000  (target: 0.90)
  F1:        0.0000
  Precision: 0.0000
  Recall:    0.0000  (important: avoid missing DDIs)
  ROC-AUC:   0.5460


In [ ]:
# --- CLINICAL MAPPING ---
TYPE_MEANINGS = {
    48: "Metabolism Inhibition (High Risk)",
    46: "Toxicity Accumulation",
    69: "QT Prolongation (Arrhythmia Risk)",
    19: "Bleeding Risk Increase",
    "default": "Unspecified Mechanism"
}

In [ ]:
def predict_cardio_interaction_safe(drug_a_id, drug_b_id, model, graph):
    """Predicts interaction using full forward pass."""
    model.eval()
    
    if drug_a_id not in graph['drug_map'] or drug_b_id not in graph['drug_map']:
        print("⚠️ Error: Drug IDs not found in database.")
        return
        
    u = graph['drug_map'][drug_a_id]
    v = graph['drug_map'][drug_b_id]
    
    query_node_indices = torch.tensor([u, v], dtype=torch.long)
    
    # Extract 1-hop subgraph
    subset, edge_index, mapping, edge_mask = k_hop_subgraph(
        node_idx=query_node_indices, 
        num_hops=1, 
        edge_index=graph['edge_index'], 
        relabel_nodes=True
    )
    
    # Map to subgraph indices
    subgraph_u = mapping[0]
    subgraph_v = mapping[1]
    query_edge = torch.tensor([[subgraph_u], [subgraph_v]], device=DEVICE)
    
    # Get features for subgraph
    x_v1 = graph['x_v1'][subset].to(DEVICE)
    x_v2 = graph['x_v2'][subset].to(DEVICE)
    x_v3 = graph['x_v3'][subset].to(DEVICE)
    edge_index = edge_index.to(DEVICE)
    
    with torch.no_grad():
        # Model returns only 2 values: (logits_bin, logits_type)
        logits_bin, logits_type = model(x_v1, x_v2, x_v3, edge_index, query_edge)
        
        prob = torch.softmax(logits_bin, dim=1)[0, 1].item()
        type_idx = torch.argmax(logits_type, dim=1).item()
        orig_type = graph['encoder'].inverse_transform([type_idx])[0]
    
    # --- REPORT GENERATION ---
    print(f"\n{'='*50}")
    print(f"❤️  Drug Interaction Analysis")
    print(f"{'='*50}")
    print(f"Drugs: {drug_a_id} + {drug_b_id}")
    print(f"Interaction Probability: {prob:.2%}")
    
    if prob > 0.5:
        desc = TYPE_MEANINGS.get(orig_type, f"Type {orig_type} (General Interaction)")
        print(f"\n🚨 ALERT: Interaction Detected")
        print(f"  • Prediction: {desc}")
        
        if orig_type in [69, 19]:
            print(f"  • CRITICAL WARNING: High-Risk for CVD patients.")
    else:
        print(f"\n✅ Assessment: Safe Combination")
    print(f"{'='*50}\n")

# Example Usage
print("Running Test Case...")
predict_cardio_interaction_safe('DB00855', 'DB00460', model, graph)

Running Test Case...


W0216 11:23:58.650000 8292 site-packages\torch\_inductor\utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
d:\AushadhiNet\.condaenv3.11\Lib\site-packages\torch\_inductor\lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(


TritonMissing: Cannot find a working triton installation. Either the package is not installed or it is too old. More information on installing Triton can be found at: https://github.com/triton-lang/triton

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"
